In [ ]:
from google.colab import drive
drive.mount('[REDACTED]')

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk import ngrams, FreqDist
from wordcloud import WordCloud
import matplotlib.pyplot as plt
#from keybert import KeyBERT
import string
import os
import pandas as pd
import re
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

In [ ]:
def read_text_files(file_path):
  with open(file_path, 'r') as file:
    return file.read()
# tokenize the text
def tokenize_text(text):
  text = word_tokenize(text)
  return text
# remove stop words
def remove_stopwords(word_tokens):
  stop_words = set(stopwords.words('english'))

  filtered_sentence = []

  for w in word_tokens:
      if w not in stop_words:
          filtered_sentence.append(w)
  return filtered_sentence
def case_normalization(text):
  return [t.lower() for t in text]
# remove punctuations
def remove_punctuation(text):
  res = [re.sub(r'[^\w\s]', '', token) for token in text if re.sub(r'[^\w\s]', '', token)]
  filtered_text = []
  for w in res:
      if len(w.strip()) > 1:
          filtered_text.append(w)
  return filtered_text
def most_freq_ngrams(text, n):
  ngrams = nltk.ngrams(text, n)
  ngram_fdist = nltk.FreqDist(ngrams)
  return ngram_fdist
# generate a word cloud
def generate_wordcloud(ngram_fdist):
    # Create a dictionary of n-grams and their frequencies
    n_grams_dict = { ' '.join(gram): freq for gram, freq in ngram_fdist.items() }

    # Create a word cloud
    wordcloud = WordCloud(width=800, height=400, background_color='white').[REDACTED](n_grams_dict)

    # Display the word cloud
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')  # Hide the axes
    plt.show()
def keyword_extractor(text, ngram, top_n):
  # Initialize the KeyBERT model
  model = KeyBERT()



  # Extract keywords
  keywords = model.extract_keywords(text,top_n=top_n, keyphrase_ngram_range=(ngram, ngram))

  # Print the keywords
  print("Keywords:")
  for keyword in keywords:
      print(keyword)
  return keywords
def find_texts(directory):
    text_files = []
    # Traverse the directory and subdirectories
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith(".txt"):
                text_files.append(os.path.join(root, file))
    return text_files
def update_dict(d1, d2):
  for k in d2.keys():
    if k in d1.keys():
      d1[k] += d2[k]
    else:
      d1[k] = d2[k]
  return d1

### Example - Word Cloud

In [ ]:
file_path = "[REDACTED]"
text = read_text_files(file_path)
text

In [ ]:
text = tokenize_text(text)

In [ ]:
text = case_normalization(text)

In [ ]:
text[:10]


In [ ]:
text = remove_stopwords(text)
text[:10]

In [ ]:
# remove numbers
# def remove_numbers(text):
#   for i in text:
#       try:
#           if str(int(float(i))).isnumeric():
#               text.remove(i)
#       except:
#           pass
#   return text

In [ ]:
# text = remove_numbers(text)

In [ ]:
text = remove_punctuation(text)

In [ ]:
text[:10]

In [ ]:
freq_dist = most_freq_ngrams(text, 1)
freq_dist

In [ ]:
bigrams = most_freq_ngrams(text, 2)
bigrams

In [ ]:
trigrams = most_freq_ngrams(text, 3)
trigrams

In [ ]:
fourgrams = most_freq_ngrams(text, 4)
fourgrams

In [ ]:
generate_wordcloud(dict(freq_dist))

In [ ]:
generate_wordcloud(dict(bigrams))


In [ ]:
generate_wordcloud(dict(trigrams))


### TF-IDF

In [ ]:
dir = ["[REDACTED]"]
text_files = []
for d in dir:
  text_files = text_files + find_texts(d)
text_files[:2]

In [ ]:
tokenized_text = []
for f in text_files:
  text = read_text_files(f)
  text = tokenize_text(text)
  text = case_normalization(text)
  text = remove_stopwords(text)
  text = remove_punctuation(text)
  tokenized_text.append(text)

In [ ]:
from pathlib import Path
text_titles = [Path(text).stem for text in text_files]

In [ ]:
def identity_tokenizer(text):
    return text
tfidf = TfidfVectorizer(tokenizer=identity_tokenizer, lowercase=False)
result = tfidf.fit_transform(tokenized_text)
tfidf_df = pd.DataFrame(result.toarray(), index=text_titles, columns=tfidf.get_feature_names_out())
# result.toarray()

In [ ]:
tfidf_df.head()

In [ ]:
tfidf = tfidf_df.reset_index().set_index(['level_0']).stack().reset_index(level=[0,1]).reset_index(drop=True).rename(columns={0:'TFIDF','level_0':'Document','level_1':'Word'})

In [ ]:
tfidf.describe()

In [ ]:
tfidf = tfidf[tfidf.TFIDF!=0]
tfidf.shape

In [ ]:
tfidf.describe()


In [ ]:
tfidf.head()

In [ ]:
tfidf = tfidf.groupby('Document', group_keys=False).apply(lambda x: x.sort_values('TFIDF', ascending=False))
tfidf.head()

In [ ]:
tfidf.to_csv('[REDACTED]', index=False)

In [ ]:
tfidf = TfidfVectorizer(tokenizer=identity_tokenizer, lowercase=False, ngram_range=(2,2))
result = tfidf.fit_transform(tokenized_text)
tfidf_df = pd.DataFrame(result.toarray(), index=text_titles, columns=tfidf.get_feature_names_out())
tfidf_df.head()


In [ ]:
tfidf_df.reset_index().set_index(['index']).stack().reset_index(level=[0,1]).reset_index(drop=True)

In [ ]:
tfidf = tfidf_df.reset_index().set_index(['index']).stack().reset_index(level=[0,1]).reset_index(drop=True).rename(columns={0:'TFIDF','index':'Document','level_1':'2 gram'})
tfidf = tfidf.groupby('Document', group_keys=False).apply(lambda x: x.sort_values('TFIDF', ascending=False))
tfidf.head()

In [ ]:
tfidf.describe()

In [ ]:
tfidf = tfidf[tfidf.TFIDF!=0]
tfidf.shape

In [ ]:
tfidf.to_csv('[REDACTED]', index=False)


In [ ]:
### for [REDACTED] documents

### All [REDACTED] Text Files

running at 11:28 AM, Oct 31

In [ ]:
dict_1gram = {}
dict_2gram = {}
dict_3gram = {}
dict_4gram = {}


In [ ]:
dir = ['[REDACTED]']
text_files = []
for d in dir:
  text_files = text_files + find_texts(d)
text_files[:2]

In [ ]:
text_files

In [ ]:
len(text_files)

In [ ]:
for f in text_files:
  text = read_text_files(f)
  text = tokenize_text(text)
  text = case_normalization(text)
  text = remove_stopwords(text)
  text = remove_punctuation(text)
  freq_dist = most_freq_ngrams(text, 1)
  bigrams = most_freq_ngrams(text, 2)
  trigrams = most_freq_ngrams(text, 3)
  dict_1gram = update_dict(dict_1gram, dict(freq_dist))
  dict_2gram = update_dict(dict_2gram, dict(bigrams))
  dict_3gram = update_dict(dict_3gram, dict(trigrams))




In [ ]:
len(dict_1gram.keys())

In [ ]:
len(dict_2gram.keys())

In [ ]:
df_1gram = pd.DataFrame()
df_1gram['words'] = dict_1gram.keys()
df_1gram['count'] = dict_1gram.values()
df_2gram = pd.DataFrame()
df_2gram['words'] = dict_2gram.keys()
df_2gram['count'] = dict_2gram.values()
df_3gram = pd.DataFrame()
df_3gram['words'] = dict_3gram.keys()
df_3gram['count'] = dict_3gram.values()


In [ ]:
df_1gram.head()


In [ ]:
df_1gram[df_1gram['words']== "('s',)"	]

In [ ]:
df_1gram = df_1gram.sort_values('count', ascending = False)
df_2gram = df_2gram.sort_values('count', ascending = False)
df_3gram = df_3gram.sort_values('count', ascending = False)


In [ ]:
df_1gram.to_csv('[REDACTED]')
df_2gram.to_csv('[REDACTED]')
df_3gram.to_csv('[REDACTED]')



In [ ]:
df_1gram = pd.read_csv('[REDACTED]')
df_2gram = pd.read_csv('[REDACTED]')
df_3gram = pd.read_csv('[REDACTED]')

In [ ]:
df_1gram.head()

In [ ]:
df_1gram[df_1gram.words == "('s',)"	]

In [ ]:
d_1gram = {}
for i in range(df_1gram.shape[0]):
  d_1gram[df_1gram.iloc[i]['words'].strip("(),'")] = df_1gram.iloc[i]['count']


In [ ]:
generate_wordcloud(d_1gram)

In [ ]:
df_2gram.iloc[i]['words'].strip("(),'").replace(',',' ').replace("'",'')

In [ ]:
d_2gram = {}
for i in range(df_2gram.shape[0]):
  d_2gram[df_2gram.iloc[i]['words'].strip("(),'").replace(',',' ').replace("'",'')] = df_2gram.iloc[i]['count']
generate_wordcloud(d_2gram)


In [ ]:
d_3gram = {}
for i in range(df_3gram.shape[0]):
  d_3gram[df_3gram.iloc[i]['words'].strip("(),'").replace(',',' ').replace("'",'')] = df_3gram.iloc[i]['count']
generate_wordcloud(d_3gram)

### [REDACTED] Text Files

In [ ]:
dict_1gram = {}
dict_2gram = {}
dict_3gram = {}
dict_4gram = {}


In [ ]:
dir = ["[REDACTED]"]
text_files = []
for d in dir:
  text_files = text_files + find_texts(d)
text_files[:2]

In [ ]:
for f in text_files:
  text = read_text_files(f)
  text = tokenize_text(text)
  text = case_normalization(text)
  text = remove_stopwords(text)
  text = remove_punctuation(text)
  freq_dist = most_freq_ngrams(text, 1)
  bigrams = most_freq_ngrams(text, 2)
  trigrams = most_freq_ngrams(text, 3)
  dict_1gram = update_dict(dict_1gram, dict(freq_dist))
  dict_2gram = update_dict(dict_2gram, dict(bigrams))
  dict_3gram = update_dict(dict_3gram, dict(trigrams))




In [ ]:
df_1gram = pd.DataFrame()
df_1gram['words'] = dict_1gram.keys()
df_1gram['count'] = dict_1gram.values()
df_2gram = pd.DataFrame()
df_2gram['words'] = dict_2gram.keys()
df_2gram['count'] = dict_2gram.values()
df_3gram = pd.DataFrame()
df_3gram['words'] = dict_3gram.keys()
df_3gram['count'] = dict_3gram.values()
df_1gram = df_1gram.sort_values('count', ascending = False)
df_2gram = df_2gram.sort_values('count', ascending = False)
df_3gram = df_3gram.sort_values('count', ascending = False)



In [ ]:
df_1gram.to_csv('[REDACTED]')
df_2gram.to_csv('[REDACTED]')
df_3gram.to_csv('[REDACTED]')



In [ ]:
df_1gram = pd.read_csv('[REDACTED]')
df_2gram = pd.read_csv('[REDACTED]')
df_3gram = pd.read_csv('[REDACTED]')

In [ ]:
d_1gram = {}
for i in range(df_1gram.shape[0]):
  d_1gram[df_1gram.iloc[i]['words'].strip("(),'")] = df_1gram.iloc[i]['count']
generate_wordcloud(d_1gram)

In [ ]:
d_2gram = {}
for i in range(df_2gram.shape[0]):
  d_2gram[df_2gram.iloc[i]['words'].strip("(),'").replace(',',' ').replace("'",'')] = df_2gram.iloc[i]['count']
generate_wordcloud(d_2gram)


In [ ]:
d_3gram = {}
for i in range(df_3gram.shape[0]):
  d_3gram[df_3gram.iloc[i]['words'].strip("(),'").replace(',',' ').replace("'",'')] = df_3gram.iloc[i]['count']
generate_wordcloud(d_3gram)

### Word Clouds - Taxonomy Specific

In [ ]:
filepath = '[REDACTED]'


In [ ]:
def [REDACTED](df):
  d = {}
  for i in range(df.shape[0]):
    d[df.iloc[i]['words'].strip("(),'").replace(',',' ').replace("'","")] = df.iloc[i]['count']
  generate_wordcloud(d)


In [ ]:
import openpyxl
import pandas as pd
import requests
from io import BytesIO

spreadsheetId = "[REDACTED]" # Please set your Spreadsheet ID.
url = "[REDACTED]" + spreadsheetId
res = requests.get(url)
data = BytesIO(res.content)
xlsx = openpyxl.load_workbook(filename=data)
# for name in xlsx.sheetnames:
#     values = pd.read_excel(data, sheet_name=name)
xlsx.sheetnames

In [ ]:
# Analytics
df = pd.read_excel(data, sheet_name='Analytics')



In [ ]:
[REDACTED](df)

In [ ]:
# Data Monetization
df = pd.read_excel(data, sheet_name='Data Monetization')
[REDACTED](df)

In [ ]:
# Techniques, Technologies, Tools
df = pd.read_excel(data, sheet_name='Techniques, Technologies, Tools')
[REDACTED](df)

In [ ]:
# Case Study
df = pd.read_excel(data, sheet_name='Case Study')
[REDACTED](df)

In [ ]:
# Standards
df = pd.read_excel(data, sheet_name='Standards')
[REDACTED](df)

In [ ]:
# User Experience
df = pd.read_excel(data, sheet_name='User Experience')
[REDACTED](df)


In [ ]:
# Industry
df = pd.read_excel(data, sheet_name='Industry')
[REDACTED](df)


In [ ]:
# Business Objectives
df = pd.read_excel(data, sheet_name='Business Objectives')
[REDACTED](df)

In [ ]:
# Data Policy
df = pd.read_excel(data, sheet_name='Data Policy')
[REDACTED](df)

In [ ]:
# Compliance
df = pd.read_excel(data, sheet_name='Compliance')
[REDACTED](df)

In [ ]:
# Communication
df = pd.read_excel(data, sheet_name='Communication')
[REDACTED](df)

In [ ]:
# Talent Development
df = pd.read_excel(data, sheet_name='Talent Development')
[REDACTED](df)

In [ ]:
# Process
df = pd.read_excel(data, sheet_name='Process')
[REDACTED](df)

In [ ]:
# Organization
df = pd.read_excel(data, sheet_name='Organization')
[REDACTED](df)

In [ ]:
# Data Privacy & Security
df = pd.read_excel(data, sheet_name='Data Privacy & Security')
[REDACTED](df)

In [ ]:
# Data Stewardship
df = pd.read_excel(data, sheet_name='Data Stewardship')
[REDACTED](df)

In [ ]:
# Data Lands[REDACTED]e
df = pd.read_excel(data, sheet_name='Data Lands[REDACTED]e')
[REDACTED](df)

In [ ]:
# Data Management
df = pd.read_excel(data, sheet_name='Data Management')
[REDACTED](df)

In [ ]:
# Data Quality
df = pd.read_excel(data, sheet_name='Data Quality')
[REDACTED](df)

In [ ]:
# Data Strategy
df = pd.read_excel(data, sheet_name='Data Strategy')
[REDACTED](df)